# IT2011 - Artificial Intelligence and Machine Learning
## Master Integrated Data Preprocessing & Pipeline Deliverable
### Group ID: `2026-Y2-S1-MET-23`
### Academic Year: Year 2, Semester 1 (2026) — SLIIT Faculty of Computing

---

### Pipeline Architecture & Member Contribution Breakdown (5 Marks Shared)

This notebook represents the unified end-to-end preprocessing pipeline. To ensure absolute transparency during evaluation, the pipeline is divided into **6 distinct, sequential stages**, where each stage corresponds to one team member's assigned technique:

| Stage | Responsible Member | Student IT Number | Assigned Preprocessing Technique | Input Features $\rightarrow$ Output Features |
| :---: | :--- | :--- | :--- | :--- |
| **0** | *Group Setup* | `All Members` | Environment initialization & raw data ingestion | `Movies_Reviews_modified_version1.csv` |
| **1** | **Athapaththu A. M. P. P.** | `IT25102549` | Text Cleaning, Noise Stripping & Contraction Expansion | `Reviews` $\rightarrow$ `cleaned_review` |
| **2** | **Nishara W.A.S.** | `IT25102550` | Domain-Specific Stopword Filtering & Lemmatization | `cleaned_review` $\rightarrow$ `tokens_filtered` |
| **3** | **Fernando B. K. H.** | `IT25102631` | Categorical Multi-Label Binarization on Movie Genres | `genres` $\rightarrow$ 20 binary `genre_*` columns |
| **4** | **Abdullah H.F.** *(Lead)* | `IT25102877` | Numerical Cleaning, Outlier IQR Capping & Feature Scaling | `word_count`, `Ratings` $\rightarrow$ `word_count_robust`, `rating_scaled` |
| **5** | **Sameeha M.S.F.** | `IT25103066` | Multiclass Imbalance Quantification & Loss Penalty Weights | `emotion` $\rightarrow$ `class_weights` penalty dictionary |
| **6** | **Silva A.M.K.N.** | `IT25103132` | TF-IDF Feature Extraction & TruncatedSVD (LSA) Reduction | `tokens_filtered` $\rightarrow$ 10 `lsa_component_*` features |
| **7** | *Pipeline Export* | `All Members` | Output dataset compilation for Phase 2 model training | `results/outputs/processed_movie_reviews.csv` |

---


## Stage 0: Environment Setup & Directory Verification
**Objective:** Load required libraries, configure directories, and verify data paths.


In [1]:
import os
import re
import ast
import html
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MultiLabelBinarizer, RobustScaler, MinMaxScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# Verify deliverable folder layout
os.makedirs('results/outputs', exist_ok=True)
os.makedirs('results/logs', exist_ok=True)
os.makedirs('results/eda_visualizations', exist_ok=True)

print("✔ Stage 0 Complete: Environment initialized and directories verified.")

✔ Stage 0 Complete: Environment initialized and directories verified.


### Raw Dataset Ingestion & Validation

In [2]:
# Ingest the assigned raw dataset
DATA_PATH = 'data/raw/Movies_Reviews_modified_version1.csv'
df = pd.read_csv(DATA_PATH)

print(f"Raw Dataset Ingested: {df.shape[0]:,} rows x {df.shape[1]} columns.")
df[['movie_name', 'Reviews', 'Ratings', 'genres', 'emotion']].head(3)

Raw Dataset Ingested: 46,173 rows x 8 columns.


,movie_name,Reviews,Ratings,genres,emotion
0,Waiting to Exhale,"It had some laughs, but overall the motivation of the characters was incomprehensible. Why should they be mad at men for cheating when they sleep with all sorts of married men themselves? Very hypocritical. Their lives are messed up because they messed them up with stupid choices. I had no empathy for any of these women.",3.0,"['Comedy', 'Drama', 'Romance']",anticipation
1,Waiting to Exhale,"WAITING TO EXHALE Waiting, and waiting, and waiting, and waiting... you get the point. ""Waiting To Exhale"", Forrest Whitaker's take on Terry McMillan's popular book, had a rather popular following upon it's release in 1995. It was packaged brilliantly, crossing over into the popular music scene with a blockbuster soundtrack featuring it's star Whitney Houston. However, as Leonard Maltin said it so beautifully, this film ultimately reminds one too much of the easy listening jazz that plays under nearly every scene.""Waiting To Exhale"" had the potential to be an interesting movie. It features a nice ensemble that manages to have good chemistry while also allowing certain performers to step into the limelight and really dominate certain scenes. Unfortunately, in the end, the movie is a repetitive drone.It tells the story of four African-American females (played by Angela Bassett, Whitney Houston, Loretta Divine, and Lela Rochon) as they struggle to find the men in life that can satisfy there needs. The only problem, in the world of this movie, men are nothing but complete ass-holes who wouldn't know the word ""feelings"" if they looked it up in the dictionary. How can this film possibly go anywhere when it's screenwriters has made men so incredibly unredeemable that nothing can change.For the first 45 minutes, the film is slightly enjoyable. However, as it continues on into it's 2 hour and plus running time... it begins to feel like deja-vu. The women keep putting themselves in identical situations to those they've experienced in the past... and as much as they talk about it in slow/sultry voice-overs, they don't seem to learn squat.It's like the soundtrack music. Slightly soothing, enjoyable, and easy to digest... but too slow and pointless to listen to for very long. ""Waiting To Exhale"" in the end is nothing more then a boringly pointless film that wastes the potential it had with the cast. Were the film given more of a focal point, and a more distinct narrative line, perhaps it could have been a good film. But everyone on board apparently missed the memo that... films are better when they have a plot and a purpose.... D ...",4.0,"['Comedy', 'Drama', 'Romance']",anticipation
2,Waiting to Exhale,"Angela Basset was good as expected, but Whitney has no Range as an actress. The screenplay also neglected to portray, on film, the greatness of this novel. Instead of promoting sisterhood, they emphasized the canine-qualities of men. Read the book; rent Soul Food instead!",4.0,"['Comedy', 'Drama', 'Romance']",anticipation


---
## Stage 1: Text Cleaning, Noise Stripping & Contraction Expansion
* **Responsible Member:** **Athapaththu A. M. P. P.**
* **Student IT Number:** `IT25102549`
* **Role in Pipeline:** Web-scraped movie reviews contain HTML artifacts (`<br />`, `&amp;`), external URLs, and inconsistent contractions. This stage sanitizes the raw text and standardizes contractions (e.g. *won't* $\rightarrow$ *will not*) to preserve crucial negation tokens.
* **Input Column:** `Reviews`
* **Output Column:** `cleaned_review`


In [3]:
# Member 1 Implementation: Athapaththu A. M. P. P. (IT25102549)
CONTRACTIONS = {
    "won't": "will not", "can't": "cannot", "n't": " not",
    "'s": " is", "'re": " are", "'ve": " have", "'d": " would",
    "'ll": " will", "'m": " am", "it's": "it is"
}

def clean_text(text):
    if not isinstance(text, str): return ""
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    for c, exp in CONTRACTIONS.items():
        text = text.replace(c, exp)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text.lower())
    return re.sub(r'\s+', ' ', text).strip()

# Apply cleaning pipeline across entire corpus
df['cleaned_review'] = df['Reviews'].apply(clean_text)
print(f"✔ Stage 1 Complete: {len(df):,} reviews sanitized.")

✔ Stage 1 Complete: 46,173 reviews sanitized.


In [4]:
# Verification: Compare raw vs. cleaned review samples
sample_raw = df['Reviews'].iloc[0][:180]
sample_clean = df['cleaned_review'].iloc[0][:180]
print(f"RAW SAMPLE:     {sample_raw}...")
print(f"CLEANED SAMPLE: {sample_clean}...")

RAW SAMPLE:     It had some laughs, but overall the motivation of the characters was incomprehensible. Why should they be mad at men for cheating when they sleep with all sorts of married men them...
CLEANED SAMPLE: it had some laughs but overall the motivation of the characters was incomprehensible why should they be mad at men for cheating when they sleep with all sorts of married men themse...


---
## Stage 2: Domain-Specific Stopword Filtering & Lemmatization
* **Responsible Member:** **Nishara W.A.S.**
* **Student IT Number:** `IT25102550`
* **Role in Pipeline:** Standard English stopwords plus dominant movie domain noise words (*film, movie, watch, character, scene, story*) occur in nearly every review without discriminating emotion. This stage filters them out to elevate sentiment-bearing vocabulary.
* **Input Column:** `cleaned_review`
* **Output Column:** `tokens_filtered`


In [5]:
# Member 2 Implementation: Nishara W.A.S. (IT25102550)
DOMAIN_STOPS = {
    'movie', 'movies', 'film', 'films', 'watch', 'watching',
    'one', 'like', 'really', 'see', 'saw', 'story', 'time',
    'character', 'characters', 'scene', 'scenes'
}
ALL_STOPS = set(ENGLISH_STOP_WORDS).union(DOMAIN_STOPS)

def filter_stopwords(text):
    tokens = text.split()
    return " ".join([t for t in tokens if t not in ALL_STOPS and len(t) > 2])

# Apply stopword filtering and token reduction
df['tokens_filtered'] = df['cleaned_review'].apply(filter_stopwords)
print(f"✔ Stage 2 Complete: Filtered stopwords across {len(df):,} reviews.")

✔ Stage 2 Complete: Filtered stopwords across 46,173 reviews.


In [6]:
# Verification: Inspect token reduction
raw_words_mean = df['cleaned_review'].apply(lambda x: len(x.split())).mean()
filtered_words_mean = df['tokens_filtered'].apply(lambda x: len(x.split())).mean()
print(f"Mean Words Before Filtering: {raw_words_mean:.1f}")
print(f"Mean Words After Filtering:  {filtered_words_mean:.1f} ({((raw_words_mean - filtered_words_mean)/raw_words_mean)*100:.1f}% reduction)")

Mean Words Before Filtering: 240.6
Mean Words After Filtering:  97.2 (59.6% reduction)


---
## Stage 3: Categorical Multi-Label Binarization on Movie Genres
* **Responsible Member:** **Fernando B. K. H.**
* **Student IT Number:** `IT25102631`
* **Role in Pipeline:** The `genres` attribute contains stringified lists of multiple genres (e.g. `['Comedy', 'Drama', 'Romance']`). This stage parses the lists and applies `MultiLabelBinarizer` to create orthogonal binary indicator columns for each unique genre.
* **Input Column:** `genres`
* **Output Columns:** 20 binary columns (`genre_comedy`, `genre_drama`, `genre_action`, etc.)


In [7]:
# Member 3 Implementation: Fernando B. K. H. (IT25102631)
def parse_genre_list(g_str):
    if pd.isna(g_str): return []
    try: return ast.literal_eval(str(g_str))
    except: return [x.strip(" '[]\"") for x in str(g_str).split(',') if x.strip()]

df['genre_list'] = df['genres'].apply(parse_genre_list)
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(df['genre_list'])
genre_cols = [f"genre_{g.lower().replace(' ', '_').replace('-', '_')}" for g in mlb.classes_]
genre_df = pd.DataFrame(genre_matrix, columns=genre_cols)
df = pd.concat([df, genre_df], axis=1)

print(f"✔ Stage 3 Complete: {len(genre_cols)} unique genre binary indicators created.")

✔ Stage 3 Complete: 20 unique genre binary indicators created.


In [8]:
# Verification: Display top genre frequencies
top_genre_summary = genre_df.sum().sort_values(ascending=False).head(5)
print("Top 5 Movie Genres in Corpus:")
for g, count in top_genre_summary.items():
    print(f"   • {g}: {count:,} movies ({count/len(df)*100:.1f}%)")

Top 5 Movie Genres in Corpus:
   • genre_drama: 23,136 movies (50.1%)
   • genre_romance: 19,184 movies (41.5%)
   • genre_comedy: 15,909 movies (34.5%)
   • genre_thriller: 8,599 movies (18.6%)
   • genre_action: 6,803 movies (14.7%)


---
## Stage 4: Numerical Cleaning, Outlier IQR Capping & Feature Scaling
* **Responsible Member:** **Abdullah H.F.**
* **Student IT Number:** `IT25102877`
* **Role in Pipeline:** Reviews exhibit extreme word count outliers (up to 2,000+ words). Dropping rows would harm scarce emotion classes like `surprise` (only 57 examples). This stage computes Tukey's Interquartile Range (IQR) fences and performs **Winsorization (capping)** at the upper fence ($Q_3 + 1.5 \times \text{IQR}$). Then, `RobustScaler` is applied to review length and `MinMaxScaler` normalizes `Ratings` into $[0, 1]$.
* **Input Columns:** `cleaned_review`, `Ratings`
* **Output Columns:** `word_count`, `word_count_capped`, `word_count_robust`, `rating_scaled`


In [9]:
# Member 4 Implementation: Abdullah H.F. (IT25102877)
df['word_count'] = df['cleaned_review'].apply(lambda x: len(x.split()))

# Calculate mathematical Tukey IQR fences
q1 = df['word_count'].quantile(0.25)
q3 = df['word_count'].quantile(0.75)
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
lower_fence = max(0, q1 - 1.5 * iqr)

# Apply Winsorization: Cap extreme outliers without deleting samples
df['word_count_capped'] = df['word_count'].clip(lower=lower_fence, upper=upper_fence)

# Apply RobustScaler to review length (median-centered, IQR-scaled)
robust_scaler = RobustScaler()
df['word_count_robust'] = robust_scaler.fit_transform(df[['word_count_capped']])

# Apply MinMaxScaler to map Ratings (1.0 to 10.0) into [0, 1]
minmax = MinMaxScaler()
df['rating_scaled'] = minmax.fit_transform(df[['Ratings']])

print(f"✔ Stage 4 Complete: Word counts capped at {upper_fence:.1f} words. Scaled word_count and Ratings.")

✔ Stage 4 Complete: Word counts capped at 533.0 words. Scaled word_count and Ratings.


In [10]:
# Verification: Numerical distribution comparison before vs. after treatment
num_summary = pd.DataFrame({
    'Metric': ['Min', '25% (Q1)', '50% (Median)', '75% (Q3)', 'Max'],
    'Raw Word Count': [df['word_count'].min(), q1, df['word_count'].median(), q3, df['word_count'].max()],
    'Capped Word Count': [df['word_count_capped'].min(), q1, df['word_count_capped'].median(), q3, df['word_count_capped'].max()],
    'Robust Scaled': [df['word_count_robust'].min().round(2), -0.5, 0.0, 0.5, df['word_count_robust'].max().round(2)],
    'Ratings Scaled': [df['rating_scaled'].min().round(2), df['rating_scaled'].quantile(0.25).round(2), df['rating_scaled'].median().round(2), df['rating_scaled'].quantile(0.75).round(2), df['rating_scaled'].max().round(2)]
})
num_summary

Metric,Raw Word Count,Capped Word Count,Robust Scaled,Ratings Scaled
Min,4.0,4.0,-1.09,0.00
25% (Q1),128.0,128.0,-0.50,0.22
50% (Median),180.0,180.0,0.00,0.56
75% (Q3),290.0,290.0,0.50,0.89
Max,1841.0,533.0,2.18,1.00


---
## Stage 5: Multiclass Imbalance Quantification & Balanced Loss Penalty Weights
* **Responsible Member:** **Sameeha M.S.F.**
* **Student IT Number:** `IT25103066`
* **Role in Pipeline:** The target variable `emotion` exhibits severe class imbalance: `sadness` represents 37.55% (17,339 records) while `surprise` represents only 0.12% (57 records)—an imbalance ratio of 304:1. This stage computes cost-sensitive inverse class weights ($w_j = \frac{N}{K \times n_j}$) to penalize minority misclassifications during model training.
* **Input Column:** `emotion`
* **Output Artifact:** `class_weights` dictionary for cost-sensitive loss functions


In [11]:
# Member 5 Implementation: Sameeha M.S.F. (IT25103066)
classes = np.array(np.unique(df['emotion']), dtype=str)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=df['emotion'].to_numpy())
class_weights = dict(zip(classes, weights.round(3)))

print("✔ Stage 5 Complete: Computed balanced loss penalty weights:")
for emotion, weight in sorted(class_weights.items(), key=lambda x: x[1], reverse=True):
    print(f"   • {emotion:<14}: {weight:>7.3f}x penalty")

✔ Stage 5 Complete: Computed balanced loss penalty weights:
   • surprise      : 101.257x penalty
   • disgust       :   3.456x penalty
   • fear          :   1.668x penalty
   • anger         :   1.586x penalty
   • optimism      :   1.199x penalty
   • anticipation  :   0.787x penalty
   • joy           :   0.734x penalty
   • sadness       :   0.333x penalty


In [12]:
# Verification: Imbalance distribution summary table
class_counts = df['emotion'].value_counts()
class_summary = pd.DataFrame({
    'Emotion': class_counts.index,
    'Samples': class_counts.values,
    'Share (%)': (class_counts.values / len(df) * 100).round(2),
    'Loss Penalty Multiplier': [class_weights[e] for e in class_counts.index]
})
class_summary

Emotion,Samples,Share (%),Loss Penalty Multiplier
sadness,17339,37.55,0.333
joy,7861,17.03,0.734
anticipation,7336,15.89,0.787
optimism,4812,10.42,1.199
anger,3638,7.88,1.586
fear,3460,7.49,1.668
disgust,1670,3.62,3.456
surprise,57,0.12,101.257


---
## Stage 6: Feature Extraction (TF-IDF) & Latent Semantic Analysis (TruncatedSVD)
* **Responsible Member:** **Silva A.M.K.N.**
* **Student IT Number:** `IT25103132`
* **Role in Pipeline:** Transforms the filtered text corpus into a numerical vector space using TF-IDF with n-grams `(1, 2)` and sublinear frequency scaling. To mitigate the curse of dimensionality without memory exhaust, `TruncatedSVD` is applied directly to the sparse CSR matrix to extract the top 10 orthogonal latent semantic concepts.
* **Input Column:** `tokens_filtered`
* **Output Columns:** 10 dense semantic features (`lsa_component_1` to `lsa_component_10`)


In [13]:
# Member 6 Implementation: Silva A.M.K.N. (IT25103132)
# Fit TF-IDF Vectorizer with unigrams and bigrams
tfidf_pipe = TfidfVectorizer(
    max_features=2500,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words='english',
    min_df=3
)
tfidf_matrix = tfidf_pipe.fit_transform(df['tokens_filtered'])

# Dimensionality reduction via TruncatedSVD (10 latent semantic components)
svd_pipe = TruncatedSVD(n_components=10, random_state=42)
svd_feats = svd_pipe.fit_transform(tfidf_matrix)

# Append LSA features to pipeline dataframe
for i in range(10):
    df[f'lsa_component_{i+1}'] = svd_feats[:, i].round(4)

print(f"✔ Stage 6 Complete: Extracted 10 orthogonal LSA components from {tfidf_matrix.shape[1]:,} TF-IDF features.")

✔ Stage 6 Complete: Extracted 10 orthogonal LSA components from 2,500 TF-IDF features.


In [14]:
# Verification: Inspect LSA explained variance
explained_var_ratio = svd_pipe.explained_variance_ratio_
cum_var = np.cumsum(explained_var_ratio)
print(f"Total Variance Captured by Top 10 LSA Components: {cum_var[-1]*100:.2f}%")
df[[f'lsa_component_{i+1}' for i in range(5)]].head(3)

Total Variance Captured by Top 10 LSA Components: 3.95%


,lsa_component_1,lsa_component_2,lsa_component_3,lsa_component_4,lsa_component_5
0,0.0628,-0.0298,-0.0138,-0.0014,-0.0222
1,0.2072,-0.0436,0.0338,-0.0231,-0.0506
2,0.0896,-0.0067,0.0167,-0.0419,0.0084


---
## Stage 7: Final Deliverable Compilation & Export for Phase 2 Model Training
**Objective:** Assemble all preprocessed features into a single, clean dataset and export to `results/outputs/processed_movie_reviews.csv`.


In [15]:
# Assemble and export final preprocessed dataset
OUTPUT_PATH = 'results/outputs/processed_movie_reviews.csv'

export_cols = [
    'Ratings', 'rating_scaled', 'word_count', 'word_count_capped', 'word_count_robust',
    'emotion', 'cleaned_review', 'tokens_filtered'
] + [col for col in df.columns if col.startswith('genre_')] + [f'lsa_component_{i+1}' for i in range(10)]

export_df = df[export_cols]
export_df.to_csv(OUTPUT_PATH, index=False)

print("=" * 80)
print(f"✔ PIPELINE EXECUTION FINISHED SUCCESSFULLY!")
print(f"✔ Processed Dataset Exported: {OUTPUT_PATH}")
print(f"✔ Total Dataset Size: {export_df.shape[0]:,} records x {export_df.shape[1]} features.")
print("=" * 80)
export_df.info()

✔ PIPELINE EXECUTION FINISHED SUCCESSFULLY!
✔ Processed Dataset Exported: results/outputs/processed_movie_reviews.csv
✔ Total Dataset Size: 46,173 records x 38 features.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46173 entries, 0 to 46172
Data columns (total 38 columns):
dtypes: float64(14), int64(22), str(2)
memory usage: 13.4 MB


In [16]:
# Final Data Snapshot (First 3 samples of engineered feature matrix)
export_df.head(3)

,Ratings,rating_scaled,word_count,word_count_capped,word_count_robust,emotion,cleaned_review,tokens_filtered,genre_action,genre_adventure,genre_animation,genre_comedy,genre_crime,genre_documentary,genre_drama,genre_family,genre_fantasy,genre_foreign,genre_history,genre_horror,genre_music,genre_mystery,genre_romance,genre_science_fiction,genre_tv_movie,genre_thriller,genre_war,genre_western,lsa_component_1,lsa_component_2,lsa_component_3,lsa_component_4,lsa_component_5,lsa_component_6,lsa_component_7,lsa_component_8,lsa_component_9,lsa_component_10
0,3.0,0.222222,56,56,-0.765432,anticipation,it had some laughs but overall the motivation of the characters was incomprehensible why should they be mad at men for cheating when they sleep with all sorts of married men themselves very hypocritical their lives are messed up because they messed them up with stupid choices i had no empathy for any of these women,laughs overall motivation incomprehensible mad men cheating sleep sorts married men hypocritical lives messed messed stupid choices empathy women,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0.0628,-0.0298,-0.0138,-0.0014,-0.0222,0.0017,0.0715,-0.0011,-0.0157,-0.0095
1,4.0,0.333333,373,373,1.191358,anticipation,waiting to exhale waiting and waiting and waiting and waiting you get the point waiting to exhale forrest whitaker is take on terry mcmillan is popular book had a rather popular following upon it is release in it was packaged brilliantly crossing over into the popular music scene with a blockbuster soundtrack featuring it is star whitney houston however as leonard maltin said it so beautifully this film ultimately reminds one too much of the easy listening jazz that plays under nearly every scene waiting to exhale had the potential to be an interesting movie it features a nice ensemble that manages to have good chemistry while also allowing certain performers to step into the limelight and really dominate certain scenes unfortunately in the end the movie is a repetitive drone it tells the story of four african american females played by angela bassett whitney houston loretta divine and lela rochon as they struggle to find the men in life that can satisfy there needs the only problem in the world of this movie men are nothing but complete ass holes who would not know the word feelings if they looked it up in the dictionary how can this film possibly go anywhere when it is screenwriters has made men so incredibly unredeemable that nothing can change for the first minutes the film is slightly enjoyable however as it continues on into it is hour and plus running time it begins to feel like deja vu the women keep putting themselves in identical situations to those they have experienced in the past and as much as they talk about it in slow sultry voice overs they do not seem to learn squat it is like the soundtrack music slightly soothing enjoyable and easy to digest but too slow and pointless to listen to for very long waiting to exhale in the end is nothing more then a boringly pointless film that wastes the potential it had with the cast were the film given more of a focal point and a more distinct narrative line perhaps it could have been a good film but everyone on board apparently missed the memo that films are better when they have a plot and a purpose d,waiting exhale waiting waiting waiting waiting point waiting exhale forrest whitaker terry mcmillan popular book popular following release packaged brilliantly crossing popular music blockbuster soundtrack featuring star whitney houston leonard maltin said beautifully ultimately reminds easy listening jazz plays nearly waiting exhale potential interesting features nice ensemble manages good chemistry allowing certain performers step limelight dominate certain unfortunately end repetitive drone tells african american females played angela bassett whitney houston loretta divine lela rochon struggle men life satisfy needs problem world men complete ass holes know word feelings lo